# rdepth 2차 사이클 — 루프×MoE (v3, 재개 버그 수정 내장)

**사용법**: 런타임 유형 **L4 GPU** 확인 → **런타임 → 모두 실행** → 드라이브 허용 팝업만 처리. 끝.

완주한 런은 자동으로 몇 초 만에 통과하고, 남은 것만 학습합니다.

In [ ]:
!nvidia-smi -L

In [ ]:
# [1] 드라이브 + 코드 준비 + 재개버그 패치 (전부 이 셀 하나에서 처리)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs(DRIVE, exist_ok=True)
zpath = f'{DRIVE}/rdepth_code_v2.zip'
if not os.path.exists(zpath):
    from google.colab import files
    print('rdepth_code_v2.zip 파일을 선택해 주세요:')
    up = files.upload()
    shutil.move(list(up)[0], zpath)
os.system(f'unzip -q -o {zpath} -d /content/rdepth')
src = open('/content/rdepth/train.py').read()
src = src.replace('map_location=dev, weights_only=False', 'map_location="cpu", weights_only=False')
open('/content/rdepth/train.py', 'w').write(src)
%cd /content/rdepth
print('코드 준비 + 패치 완료:', 'map_location="cpu"' in src)

In [ ]:
# [2] 데이터 (드라이브 캐시에서 ~1분)
%cd /content/rdepth
import os
DRIVE = '/content/drive/MyDrive/rdepth_out'
os.makedirs('data', exist_ok=True)
if os.path.exists(f'{DRIVE}/train.bin'):
    print('Drive 캐시에서 복사...')
    !cp {DRIVE}/tok4096.json {DRIVE}/val.bin {DRIVE}/train.bin data/
else:
    !python prepare_data.py
    !cp data/tok4096.json data/val.bin data/train.bin {DRIVE}/
print('데이터 준비 완료')

In [ ]:
# [3] GPU에 맞는 dtype 자동 선택
import torch, os
os.environ['RDEPTH_OUT'] = '/content/drive/MyDrive/rdepth_out'
cap = torch.cuda.get_device_capability()
DTYPE = 'bf16' if cap[0] >= 8 else 'fp16'
print('GPU =', torch.cuda.get_device_name(0), cap, '| dtype =', DTYPE)

In [ ]:
# [4-1] moe-small — 완주됨: 몇 초 만에 통과 예상
%cd /content/rdepth
!python train.py --run moe-small --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --resume

In [ ]:
# [4-2] moe-loop-st(sticky) — 완주됨: 몇 초 만에 통과 예상
%cd /content/rdepth
!python train.py --run moe-loop-st --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --resume

In [ ]:
# [4-3] moe-loop-rr — 완주 또는 끝부분만 재개
%cd /content/rdepth
!python train.py --run moe-loop-rr --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --resume

In [ ]:
# [4-4] moe-large — 실제 학습 (L4에서 ~40분)
%cd /content/rdepth
!python train.py --run moe-large --max-tokens 100000000 --dtype {DTYPE} --micro-batch 16 --resume

In [ ]:
# [5] 최종 판정
import csv, os
out = '/content/drive/MyDrive/rdepth_out'
rows, uniq = {}, {}
for r in ['moe-small', 'moe-loop-st', 'moe-loop-rr', 'moe-large']:
    p = f'{out}/logs/{r}.csv'
    if os.path.exists(p):
        with open(p) as f: data = list(csv.DictReader(f))
        if data:
            rows[r] = min(float(d['val_loss']) for d in data)
            uniq[r] = data[-1].get('uniq_exp', '')
print('최저 val loss:', rows)
print('uniq_exp:', uniq)
if len(rows) == 4:
    s, st, rr, g = rows['moe-small'], rows['moe-loop-st'], rows['moe-loop-rr'], rows['moe-large']
    best = min(st, rr)
    rec = (s - best) / (s - g) * 100 if s != g else float('nan')
    print(f"kill-gate(loop<small): {'PASS' if best < s else 'FAIL'}   최선 회복률 {rec:.1f}%")
    print(f"sticky {st:.4f} vs rr {rr:.4f} (Δ {st-rr:+.4f}) | 접촉 st {uniq.get('moe-loop-st')} vs rr {uniq.get('moe-loop-rr')}")